Download repo

In [ ]:
# 1) Clone + branch con build_road_quality_rqz.py
!git clone https://github.com/andrea444460/gnss_gpu_los_nlos.git
%cd /content/gnss_gpu_los_nlos
!git fetch origin cursor/build-road-quality-rqz-28e6
!git checkout cursor/build-road-quality-rqz-28e6
!git log -1 --oneline

# Colab recente: installa python3.10 completo + pip user-site (evita numpy Debian rotto)
!apt-get update
!apt-get install -y python3.10 python3.10-dev python3.10-distutils python3.10-venv
!curl -sS https://bootstrap.pypa.io/get-pip.py -o /tmp/get-pip.py
!python3.10 /tmp/get-pip.py --break-system-packages
!python3.10 -m pip install -U pip --break-system-packages
!python3.10 -m pip install --ignore-installed --force-reinstall --no-cache-dir \
  numpy requests rasterio pyproj folium matplotlib branca --break-system-packages
!python3.10 -c "import numpy; print('numpy', numpy.__version__, numpy.__file__)"


VAR

In [ ]:
# Stesso schema del tuo RayTracingCityMap.ipynb
# BBox Genova (routing GUI). Per Cipro/TERF rimetti i tuoi 34.64 / 32.97 ...
NAV_RINEX = "/content/gnss_gpu_los_nlos/BRDC00IGS_R_20252600000_01D_MN.rnx"
OUT_DIR = "/content/gnss_gpu_los_nlos/experiments/results"
SOUTH, WEST, NORTH, EAST = 44.3850, 8.8800, 44.4450, 8.9800

import os
os.makedirs(OUT_DIR, exist_ok=True)
print(OUT_DIR, SOUTH, WEST, NORTH, EAST)

BUILD

In [ ]:
%cd /content/gnss_gpu_los_nlos/

!apt-get update
!apt-get install -y python3.10 python3.10-dev python3.10-distutils pybind11-dev cmake

# Ripara numpy per python3.10 (Debian dist-packages e' spesso rotto)
!python3.10 -m pip install --ignore-installed --force-reinstall --no-cache-dir \
  numpy requests rasterio pyproj --break-system-packages

!rm -rf build
!mkdir -p build
%cd build

!cmake .. \
  -DPYTHON_EXECUTABLE=/usr/bin/python3.10 \
  -DPYTHON_INCLUDE_DIR=/usr/include/python3.10 \
  -DPYTHON_LIBRARY=/usr/lib/x86_64-linux-gnu/libpython3.10.so \
  -DCMAKE_CUDA_ARCHITECTURES=75

!make -j$(nproc)

%cd /content/gnss_gpu_los_nlos/
!find build -name "*.so" -exec cp -f {} python/gnss_gpu/ \;
!ls python/gnss_gpu/*.so

%env PYTHONPATH=/content/gnss_gpu_los_nlos/python
!PYTHONPATH=/content/gnss_gpu_los_nlos/python python3.10 -c "import numpy, gnss_gpu._bvh; print('BVH OK', numpy.__version__)"


DEM

In [ ]:
import sys
import os

module_path = '/content/gnss_gpu_los_nlos/python'
if module_path not in sys.path:
    sys.path.append(module_path)

from gnss_gpu.io.dem_download import download_dem_for_bbox

out_tif = f"{OUT_DIR}/genova_area_dem.tif"
os.makedirs(OUT_DIR, exist_ok=True)

dem_path, ntiles = download_dem_for_bbox(
    south=SOUTH,
    west=WEST,
    north=NORTH,
    east=EAST,
    output_tif=out_tif,
    zoom=12,
)
print(f"DEM saved: {dem_path} (tiles={ntiles})")

EDIFICI OSM -> MESH

In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"
os.environ["SOUTH"] = str(SOUTH)
os.environ["WEST"] = str(WEST)
os.environ["NORTH"] = str(NORTH)
os.environ["EAST"] = str(EAST)
os.environ["OUT_DIR"] = OUT_DIR

!PYTHONPATH=/content/gnss_gpu_los_nlos/python python3.10 experiments/fetch_osm_buildings_bbox.py \
  --south $SOUTH --west $WEST --north $NORTH --east $EAST \
  --cache-json "$OUT_DIR/genova_osm_cache.json" \
  --out-triangles "$OUT_DIR/genova_osm_triangles.npy" \
  --levels-height-m 3.0 \
  --dem-path "$OUT_DIR/genova_area_dem.tif" \
  --default-height-m 10.0 \
  --base-alt-m 0.0

NAV BRDC

In [ ]:
import os, gzip, urllib.request, re
from urllib.error import HTTPError
from datetime import date, timedelta
from pathlib import Path

match = re.search(r'(\d{4})(\d{3})', Path(NAV_RINEX).name)
if not match:
    raise ValueError(f"Could not parse year/DOY from NAV_RINEX: {NAV_RINEX}")
year = int(match.group(1))
doy = int(match.group(2))
target_date = date(year, 1, 1) + timedelta(doy - 1)
NAV_DIR = Path(NAV_RINEX).parent
os.makedirs(NAV_DIR, exist_ok=True)
needed = {target_date}
print("Need NAV for dates:", sorted(needed))

def get_download_urls(d: date, expected_nav_filepath: str):
    doy = d.timetuple().tm_yday
    base_filename = Path(expected_nav_filepath).name
    return [
        f"https://igs.bkg.bund.de/root_ftp/IGS/BRDC/{d.year}/{doy:03d}/{base_filename}.gz",
        f"https://igs.bkg.bund.de/root_ftp/IGS/BRDC/{d.year}/{doy:03d}/{base_filename}.Z",
        f"https://igs.bkg.bund.de/root_ftp/IGS/BRDC/{d.year}/{doy:03d}/BRDC00IGS_R_{d.year}{doy:03d}0000_01D_MN.rnx.gz",
        f"https://igs.bkg.bund.de/root_ftp/IGS/BRDC/{d.year}/{doy:03d}/BRDC00WRD_R_{d.year}{doy:03d}0000_01D_MN.rnx.gz",
        f"https://igs.bkg.bund.de/root_ftp/EUREF/BRDC/{d.year}/{doy:03d}/BRDC00WRD_R_{d.year}{doy:03d}0000_01D_MN.rnx.gz",
    ]

for d in sorted(needed):
    out = Path(NAV_RINEX)
    if out.exists() and out.stat().st_size > 0:
        print("reuse", out)
        continue
    success = False
    for url in get_download_urls(d, NAV_RINEX):
        print(f"download {url}")
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (gnss_gpu/nav-fetch)"})
        try:
            with urllib.request.urlopen(req, timeout=180) as r:
                raw = r.read()
            if url.endswith(".Z"):
                temp_z_path = out.with_suffix('.Z')
                temp_z_path.write_bytes(raw)
                os.system(f"zcat {temp_z_path} > {out}")
                temp_z_path.unlink(missing_ok=True)
            elif url.endswith(".gz"):
                out.write_bytes(gzip.decompress(raw))
            else:
                out.write_bytes(raw)
            print("-> saved", out.name)
            success = True
            break
        except HTTPError as e:
            print(f"   [HTTP {e.code}]")
        except Exception as e:
            print(f"   [Errore: {e}]")
    if not success:
        print(f"!!! ATTENZIONE: NAV non trovato per {d}")

LOS/NLOS → .rqz (tutte le strade carrozzabili)


In [ ]:
# Tutte le strade carrozzabili (no --arterial-only). Piu' punti → piu' lento.
import os
os.environ["SOUTH"] = str(SOUTH)
os.environ["WEST"] = str(WEST)
os.environ["NORTH"] = str(NORTH)
os.environ["EAST"] = str(EAST)
os.environ["OUT_DIR"] = OUT_DIR
os.environ["NAV_RINEX"] = NAV_RINEX

%env PYTHONPATH=/content/gnss_gpu_los_nlos/python

!PYTHONPATH=/content/gnss_gpu_los_nlos/python python3.10 experiments/build_road_quality_rqz.py \
  --south $SOUTH --west $WEST --north $NORTH --east $EAST \
  --triangles-npy "$OUT_DIR/genova_osm_triangles.npy" \
  --nav "$NAV_RINEX" \
  --out "$OUT_DIR/genova_quality_24h_all.rqz" \
  --step-m 30 \
  --dt-s 60 \
  --duration-s 86400 \
  --rx-alt-mode dem \
  --tile-size-m 12000 \
  --eph-batch-chunk 32 \
  --point-batch-chunk 256 \
  --dem-auto-download \
  --dem-auto-out "$OUT_DIR/genova_area_dem.tif" \
  --metrics-csv "$OUT_DIR/metrics_rqz_all.csv"

!ls -lh "$OUT_DIR/genova_quality_24h_all.rqz" "$OUT_DIR/metrics_rqz_all.csv"


Download pack

In [ ]:
from google.colab import files
files.download(f"{OUT_DIR}/genova_quality_24h_all.rqz")
